<a href="https://colab.research.google.com/github/pragatiagarwal802-wq/surat-ev-infrastructure-analysis/blob/main/surat_ev_stations_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install osmnx
!pip install h3
import osmnx as ox
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import h3

# Define your city/area
place_name = "Surat, India"

# 1. Collect Competitors (Existing EV Stations)
print("Fetching competitor data...")
competitors = ox.features_from_place(place_name, tags={'amenity': 'charging_station'})

# 2. Collect Grid Proxy (Substations and Power Lines)
# As an EE student, proximity to these reduces your CAPEX (installation cost)
print("Fetching power infrastructure...")
power_grid = ox.features_from_place(place_name, tags={'power': ['substation', 'line', 'transformer']})

# 3. Collect Flood Proxy (Water Bodies)
print("Fetching water bodies for flood risk...")
water = ox.features_from_place(place_name, tags={'natural': 'water', 'waterway': True})

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.4 MB/s eta 0:00:00
Fetching competitor data...


InsufficientResponseError: No matching features. Check query location, tags, and log.

In [ ]:
import osmnx as ox
import pandas as pd
import geopandas as gpd

# Define the location precisely
place_name = "Surat, Gujarat, India"

# 1. EV STATIONS (Extended Tags)
# We look for 'charging_station' and also 'fuel' stations (which often add EV points)
print("Fetching EV and Fuel data...")
ev_tags = {'amenity': ['charging_station', 'fuel'], 'brand': True}
competitors = ox.features_from_place(place_name, tags=ev_tags)

# 2. POWER INFRASTRUCTURE (EE Focus)
# Adding 'tower' and 'substation' specifically
print("Fetching power infrastructure...")
power_tags = {
    'power': ['substation', 'transformer', 'line', 'tower', 'plant'],
    'industrial': 'substation'
}
power_grid = ox.features_from_place(place_name, tags=power_tags)

# 3. FLOOD PROXY (Hydrology)
print("Fetching hydrology data...")
water_tags = {'natural': 'water', 'waterway': True, 'basin': True}
water = ox.features_from_place(place_name, tags=water_tags)

# 4. TRAFFIC GENERATORS (Points of Interest)
# Traffic isn't just roads; it's WHERE people go.
print("Fetching traffic generators...")
poi_tags = {'amenity': ['mall', 'hospital', 'university', 'bus_station']}
pois = ox.features_from_place(place_name, tags=poi_tags)

print("Data collection complete!")

Fetching EV and Fuel data...
Fetching power infrastructure...
Fetching hydrology data...
Fetching traffic generators...
Data collection complete!


In [ ]:
import osmnx as ox
import h3
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon

# 1. Get Surat boundary
boundary_gdf = ox.geocode_to_gdf("Surat, Gujarat, India")
surat_poly = boundary_gdf.geometry.iloc[0]
print(f"Surat bounds: {surat_poly.bounds}")

# 2. EXTRACT COORDINATES CORRECTLY (lat/lng order)
# Not directly used by h3.polygon_to_cells when passing shapely object, but kept for context if needed later.
coords = list(surat_poly.exterior.coords)  # [(lon, lat), ...]
h3_coords_prepared = [[lat, lng] for lng, lat in coords] # Converts to [[lat, lng], ...]

print(f"First 3 coords: {h3_coords_prepared[:3]}")  # Verify lat/lng order

# 3. Generate H3 grid
# h3.polygon_to_cells expects a GeoJSON-like dictionary for a Polygon or MultiPolygon
# We convert the shapely Polygon to its GeoJSON representation.
hexagons = h3.polygon_to_cells(surat_poly.__geo_interface__, 9)
print(f"✅ Generated {len(hexagons)} hexagons")

# 4. Convert to polygons
hex_polys = []
for hex_id in hexagons[:100]:  # First 100 for speed to check generation
    boundary = h3.cell_to_boundary(hex_id, 0)  # Returns [(lat,lng), ...]
    # Convert back to shapely: lon/lat order
    poly_coords = [(lng, lat) for lat, lng in boundary]
    hex_polys.append(Polygon(poly_coords))

# 5. Save grid
h3_grid = gpd.GeoDataFrame({
    'hex_id': hexagons[:100],
    'geometry': hex_polys
}, crs="EPSG:4326")

h3_grid.to_file("surat_h3_grid.geojson", driver='GeoJSON')
print(f"✅ Saved {len(h3_grid)} hexagons to surat_h3_grid.geojson")

Surat bounds: (72.5896847, 20.8155902, 73.6990386, 21.5692623)
First 3 coords: [[21.2931731, 72.5896847], [21.2912937, 72.5897974], [21.2910588, 72.5898993]]


ValueError: Unrecognized type: <class 'dict'>

In [ ]:
# Load your local census and roads data
# Replace 'your_file_name.geojson' with your actual file paths
census = gpd.read_file('surat_census_data.json')
major_roads = gpd.read_file('surat_major_roads.shp')

# Ensure everything is in the same projection (CRS)
h3_grid = h3_grid.to_crs(epsg=3857) # Metric projection for distance calculations
competitors = competitors.to_crs(epsg=3857)
power_grid = power_grid.to_crs(epsg=3857)
water = water.to_crs(epsg=3857)
census = census.to_crs(epsg=3857)

In [ ]:
import h3
import osmnx as ox

print("h3 version:", getattr(h3, "__version__", "unknown"))
print("Has polygon_to_cells:", hasattr(h3, "polygon_to_cells"))
print("Has h3shape_to_cells:", hasattr(h3, "h3shape_to_cells"))
print("Has LatLngPoly:", hasattr(h3, "LatLngPoly"))

boundary_gdf = ox.geocode_to_gdf("Surat, Gujarat, India")
print("Boundary rows:", len(boundary_gdf))
print("Geometry type:", boundary_gdf.geometry.iloc[0].geom_type)


h3 version: 4.4.2
Has polygon_to_cells: True
Has h3shape_to_cells: True
Has LatLngPoly: True
Boundary rows: 1
Geometry type: Polygon


In [ ]:
import osmnx as ox
import h3
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon

# 1. Get Surat boundary
boundary_gdf = ox.geocode_to_gdf("Surat, Gujarat, India")
surat_poly = boundary_gdf.geometry.iloc[0]

# 2. Prepare for h3.LatLngPoly
# LatLngPoly expects exterior as a list of (lat, lng) tuples
exterior_ring_ll = [(lat, lng) for lng, lat in surat_poly.exterior.coords]

# And holes as a list of lists of (lat, lng) tuples
interior_rings_ll = []
for interior in surat_poly.interiors:
    interior_rings_ll.append([(lat, lng) for lng, lat in interior.coords])

# 3. Create LatLngPoly object
# LatLngPoly expects exterior as the first argument and holes as the second (if any)
if interior_rings_ll:
    h3_poly_obj = h3.LatLngPoly(exterior_ring_ll, interior_rings_ll)
else:
    h3_poly_obj = h3.LatLngPoly(exterior_ring_ll)

# 4. Generate hexagons
# Pass the LatLngPoly object directly
hexagons = h3.polygon_to_cells(h3_poly_obj, 9)
print(f"✅ Generated {len(hexagons)} hexagons")

# 5. Convert to GeoDataFrame
hex_geoms = []
for hex_id in hexagons:
    # v4.4.2: cell_to_boundary returns [[lat, lng]]
    boundary = h3.cell_to_boundary(hex_id)
    # Convert to shapely: [(lng, lat)]
    shapely_coords = [(lng, lat) for lat, lng in boundary]
    hex_geoms.append(Polygon(shapely_coords))

# 6. Create & save grid
h3_grid = gpd.GeoDataFrame({
    'hex_id': list(hexagons),
    'geometry': hex_geoms
}, crs="EPSG:4326")

h3_grid.to_file("surat_h3_grid_res9.geojson", driver='GeoJSON')
print(f"✅ SAVED {len(h3_grid)} hexagons to surat_h3_grid_res9.geojson")

✅ Generated 45881 hexagons
✅ SAVED 45881 hexagons to surat_h3_grid_res9.geojson


In [ ]:
import geopandas as gpd
from shapely.geometry import Point

surat = gpd.read_file("surat_metro_boundary.geojson")

flood_low = gpd.GeoDataFrame(
    {'risk': [0.8, 0.9]},
    geometry=[
        Point(72.83, 21.18).buffer(0.008),
        Point(72.84, 21.20).buffer(0.006)
    ],
    crs="EPSG:4326"
).clip(surat)

substations = gpd.GeoDataFrame(
    {'id': [1, 2, 3, 4]},
    geometry=[
        Point(72.82, 21.17),
        Point(72.88, 21.22),
        Point(72.81, 21.15),
        Point(72.85, 21.19)
    ],
    crs="EPSG:4326"
).clip(surat)

print("Flood polygons:", len(flood_low))
print("Substations:", len(substations))

flood_low.to_file("surat_flood_risk.geojson", driver="GeoJSON")
substations.to_file("surat_grid_substations.geojson", driver="GeoJSON")
print("Files saved.")

Flood polygons: 2
Substations: 4
Files saved.


In [ ]:
import geopandas as gpd
import pandas as pd

# 1. Load ALL files SAFELY
print("🔄 Loading Surat datasets...")
files = {
    'boundary': "surat_metro_boundary.geojson",
    'roads': "/content/surat_major_roads (1).geojson",
    'census': "surat_census_centroids.geojson",
    'flood': "surat_flood_risk.geojson",
    'substations': "surat_grid_substations.geojson",
    'h3_grid': "/content/surat_h3_grid_res9 (1).geojson"
}

data = {}
for name, path in files.items():
    data[name] = gpd.read_file(path)
    print(f"  {name}: {len(data[name])} features")
    print(f"    Columns: {list(data[name].columns)}")

# 2. Base grid (H3 hexagons)
h3_base = data['h3_grid'].copy()
print(f"\n📊 Base grid: {len(h3_base)} hexagons")

# 3. SAFE SPATIAL JOINS
print("\n🔗 Spatial aggregation...")

# Population per hexagon
h3_pop = gpd.sjoin(h3_base[['geometry', 'hex_id']], data['census'], predicate='intersects')
if 'population' in h3_pop.columns:
    pop_agg = h3_pop.groupby('hex_id')['population'].sum().reset_index()
    h3_base = h3_base.merge(pop_agg, on='hex_id', how='left').fillna(0)
    print(f"  Population: {len(pop_agg)} hexagons with data")
else:
    h3_base['population'] = 0
    print("  Population: No population column found")

# Road length per hexagon (SAFE)
if len(data['roads']) > 0:
    print(f"  Roads columns: {list(data['roads'].columns)}")
    # Try common road length columns
    length_col = None
    for col in ['length', 'distance', 'shape_len', 'Len']:
        if col in data['roads'].columns:
            length_col = col
            break

    if length_col:
        h3_roads = gpd.sjoin(h3_base[['geometry', 'hex_id']], data['roads'], predicate='intersects')
        road_agg = h3_roads.groupby('hex_id')[length_col].sum().reset_index()
        h3_base = h3_base.merge(road_agg, on='hex_id', how='left').fillna(0)
        h3_base['road_km'] = h3_base[length_col] / 1000
        h3_base = h3_base.drop(length_col, axis=1)
        print(f"  Roads: {len(road_agg)} hexagons with road data")
    else:
        h3_base['road_km'] = 0
        print("  Roads: No length column found")
else:
    h3_base['road_km'] = 0
    print("  Roads: Empty roads file")

# Flood risk (max)
h3_flood = gpd.sjoin(h3_base[['geometry', 'hex_id']], data['flood'], predicate='intersects')
if 'risk' in h3_flood.columns:
    flood_agg = h3_flood.groupby('hex_id')['risk'].max().reset_index()
    h3_base = h3_base.merge(flood_agg, on='hex_id', how='left').fillna(0)
    print(f"  Flood: {len(flood_agg)} hexagons affected")
else:
    h3_base['risk'] = 0
    print("  Flood: No risk column found")

# Distance to substation
h3_base['dist_nearest_substation_km'] = h3_base.geometry.apply(
    lambda x: data['substations'].geometry.distance(x).min() * 111
)

# 4. FINAL DATASET
final_dataset = h3_base[[
    'hex_id', 'population', 'road_km', 'risk', 'dist_nearest_substation_km', 'geometry'
]].rename(columns={'risk': 'flood_risk'})

print(f"\n✅ COMBINED DATASET: {len(final_dataset)} rows")
print(final_dataset[['hex_id', 'population', 'road_km', 'flood_risk', 'dist_nearest_substation_km']].head())

# 5. EXPORT
final_dataset.to_file("surat_ev_combined_dataset.geojson", driver='GeoJSON')
final_dataset.drop(columns='geometry').to_csv("surat_ev_dataset_table.csv", index=False)

print("\n🎯 FILES READY FOR DATA SCIENTIST:")
print("   surat_ev_combined_dataset.geojson")
print("   surat_ev_dataset_table.csv")

🔄 Loading Surat datasets...
  boundary: 1 features
    Columns: ['bbox_west', 'bbox_south', 'bbox_east', 'bbox_north', 'place_id', 'osm_type', 'osm_id', 'lat', 'lon', 'class', 'type', 'place_rank', 'importance', 'addresstype', 'name', 'display_name', 'geometry']
  roads: 500 features
    Columns: ['highway', 'length', 'geometry']
  census: 50 features
    Columns: ['ward_id', 'population', 'geometry']
  flood: 2 features
    Columns: ['risk', 'geometry']
  substations: 4 features
    Columns: ['id', 'geometry']
  h3_grid: 45881 features
    Columns: ['hex_id', 'geometry']

📊 Base grid: 45881 hexagons

🔗 Spatial aggregation...
  Population: 50 hexagons with data
  Roads columns: ['highway', 'length', 'geometry']
  Roads: 256 hexagons with road data
  Flood: 59 hexagons affected


/tmp/ipykernel_870/1899595386.py:74: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  lambda x: data['substations'].geometry.distance(x).min() * 111



✅ COMBINED DATASET: 45881 rows
            hex_id  population  road_km  flood_risk  \
0  8942d98f60fffff         0.0      0.0         0.0   
1  8942dd6540fffff         0.0      0.0         0.0   
2  8942d9174a7ffff         0.0      0.0         0.0   
3  8942d9b99a3ffff         0.0      0.0         0.0   
4  8942d9b198bffff         0.0      0.0         0.0   

   dist_nearest_substation_km  
0                   12.998066  
1                   27.172575  
2                   28.213880  
3                   45.190584  
4                   50.814040  

🎯 FILES READY FOR DATA SCIENTIST:
   surat_ev_combined_dataset.geojson
   surat_ev_dataset_table.csv


In [ ]:
pip install pandas geopandas shapely h3 osmnx

In [ ]:
import pandas as pd
import geopandas as gpd
import h3
import osmnx as ox
from shapely.geometry import Polygon
import warnings

# Suppress minor warnings from spatial operations
warnings.filterwarnings('ignore')

# 1. Load your dataset
print("Loading dataset...")
df = pd.read_csv('output.csv')

# 2. Define a function to convert Hex IDs to Polygons
def hex_to_polygon(hex_id):
    """Converts an H3 hex string into a Shapely Polygon."""
    # Handle both h3 version 3 and version 4 APIs
    try:
        # h3 v4
        boundary = h3.cell_to_boundary(hex_id)
        # Convert (lat, lon) to (lon, lat) for GeoPandas
        boundary = [(lon, lat) for lat, lon in boundary]
    except AttributeError:
        # h3 v3 fallback
        boundary = h3.h3_to_geo_boundary(hex_id, geo_json=True)

    return Polygon(boundary)

# 3. Create a GeoDataFrame from your hex IDs
print("Converting Hex IDs to spatial polygons...")
df['geometry'] = df['hex_id'].apply(hex_to_polygon)
# EPSG:4326 is the standard coordinate system for GPS (Lat/Lon)
gdf_hex = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

# 4. Fetch the road network from OpenStreetMap
print("Fetching OpenStreetMap road network for the area (this may take a minute)...")
# We get the bounding box of all your hexagons combined to download the roads once
combined_boundary = gdf_hex.unary_union
# network_type='drive' ensures we only get roads vehicles can use (ignores walking paths)
G = ox.graph_from_polygon(combined_boundary, network_type='drive')

# Convert the OSM graph into a GeoDataFrame of lines (edges)
nodes, edges = ox.graph_to_gdfs(G)

# 5. Project to a local UTM CRS for accurate distance calculation
print("Projecting coordinate systems for accurate metric calculation...")
# This automatically finds the best metric coordinate system for your specific global location
utm_crs = edges.estimate_utm_crs()

gdf_hex_utm = gdf_hex.to_crs(utm_crs)
edges_utm = edges.to_crs(utm_crs)

# 6. Perform the Spatial Intersection
print("Calculating road intersections within each hexagon...")
# This cuts the road lines exactly at the borders of your hexagons
intersection = gpd.overlay(edges_utm, gdf_hex_utm, how='intersection')

# Calculate the length of these cut road segments in kilometers
intersection['calc_road_km'] = intersection.geometry.length / 1000

# Group by the hex_id and sum up the road lengths
road_lengths = intersection.groupby('hex_id')['calc_road_km'].sum().reset_index()

# 7. Merge the calculated lengths back into the original dataframe
print("Finalizing dataset...")
# Merge the results
df = df.merge(road_lengths, on='hex_id', how='left')

# Replace the old 0s with the new calculated values, filling any empty hexes with 0
df['road_km'] = df['calc_road_km'].fillna(0)

# Clean up temporary columns
df = df.drop(columns=['geometry', 'calc_road_km'])

# 8. Save the corrected dataset
output_filename = "output_corrected_roads.csv"
df.to_csv(output_filename, index=False)
print(f"Success! Corrected data saved to {output_filename}")

Loading dataset...
Converting Hex IDs to spatial polygons...
Fetching OpenStreetMap road network for the area (this may take a minute)...
Projecting coordinate systems for accurate metric calculation...
Calculating road intersections within each hexagon...
Finalizing dataset...
Success! Corrected data saved to output_corrected_roads.csv


In [1]:
# 1. Install necessary spatial libraries
!pip install geopandas rasterstats h3 shapely

import pandas as pd
import geopandas as gpd
import h3
from shapely.geometry import shape, Polygon
from rasterstats import zonal_stats
import os

# 2. Load your hex data
# Ensure 'output_corrected_roads.csv' and 'india_pop.tif' are uploaded to Colab
file_path = 'output_corrected_roads.csv'
raster_path = 'india_pop.tif'

if not os.path.exists(file_path) or not os.path.exists(raster_path):
    print("Error: Please ensure both 'output_corrected_roads.csv' and 'india_pop.tif' are uploaded to your Colab files.")
else:
    df = pd.read_csv(file_path)

    # 3. Convert Hex IDs to Polygons
    # This creates the physical boundaries for each hex_id in your file
    def hex_to_polygon(hex_id):
        # Use h3.cell_to_boundary for h3 v4.x
        boundary = h3.cell_to_boundary(hex_id)
        # h3.cell_to_boundary returns [[lat, lng], ...], convert to [(lng, lat), ...] for shapely.Polygon
        return Polygon([(lng, lat) for lat, lng in boundary])

    print("Converting Hex IDs to geometries...")
    df['geometry'] = df['hex_id'].apply(hex_to_polygon)

    # Create a GeoDataFrame
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

    # 4. Extract Population Data (Zonal Statistics)
    # This 'sum' function adds up all population pixels inside each hex
    print("Extracting population data from raster (this may take a minute)...")
    stats = zonal_stats(gdf, raster_path, stats="sum")

    # 5. Assign extracted values back to the population column
    # If a hex is in an area with no data, we assign 0
    df['population'] = [s['sum'] if s['sum'] is not None else 0 for s in stats]

    # 6. Save the final updated file
    output_name = 'surat_hex_data_with_population.csv'
    df.drop(columns=['geometry']).to_csv(output_name, index=False)

    print(f"Success! Data saved to: {output_name}")
    print(df[['hex_id', 'population']].head()) # Show top results


Converting Hex IDs to geometries...
Extracting population data from raster (this may take a minute)...


KeyboardInterrupt: 

In [4]:
import pandas as pd
import h3
from tqdm import tqdm

# 1. Load data
df = pd.read_csv('surat_hex_data_with_population.csv')

# 2. Population lookup (hex_id string → population)
# In h3 v4.x, functions like k_ring generally work directly with string H3 IDs.
# So, we can use the original 'hex_id' column for lookup.
pop_lookup = dict(zip(df['hex_id'], df['population']))

def get_buffer_population(hex_id_str, k=14):  # k=14 ≈ 5km at res9
    """Total population in k-ring buffer (center + neighbors)"""
    try:
        # H3 v4+: grid_disk() | H3 v3: k_ring()
        # The k_ring function expects string H3 IDs in h3 v4.x.
        # Given previous errors, we'll try h3.grid_disk as the preferred v4 method.
        if hasattr(h3, 'grid_disk'):
            neighbor_hexes = h3.grid_disk(hex_id_str, k)
        else:
            # Fallback to k_ring if grid_disk is not found (though unlikely for v4.4.2)
            neighbor_hexes = h3.k_ring(hex_id_str, k)

        total_pop = sum(pop_lookup.get(h_neighbor, 0) for h_neighbor in neighbor_hexes)
        return total_pop
    except Exception as e:
        print(f"Error for hex {hex_id_str}: {e}")
        return 0

# 3. Calculate with progress bar
print("🔄 Calculating 5km buffer population...")
tqdm.pandas(desc="Hexes")
# Apply the function directly to the 'hex_id' column (which contains strings)
df['population_5km_buffer'] = df['hex_id'].progress_apply(get_buffer_population)

# 4. Save
# No 'hex_int' column to drop as it's no longer created
df[['hex_id', 'population', 'population_5km_buffer', 'road_km']].to_csv(
    'surat_ev_analysis_phase2.csv', index=False)
print("✅ COMPLETE! Check 'population_5km_buffer' column.")
print("\nSample results:")
print(df[['hex_id', 'population', 'population_5km_buffer']].head())
print(f"\nBuffer stats: {df['population_5km_buffer'].describe()}")

🔄 Calculating 5km buffer population...


Hexes: 100%|██████████| 45881/45881 [00:37<00:00, 1225.92it/s]


✅ COMPLETE! Check 'population_5km_buffer' column.

Sample results:
            hex_id  population  population_5km_buffer
0  8942d98f60fffff         0.0           61998.566605
1  8942dd6540fffff         0.0            4483.623503
2  8942d9174a7ffff         0.0           54981.328888
3  8942d9b99a3ffff         0.0           40620.524384
4  8942d9b198bffff         0.0           22955.571945

Buffer stats: count    4.588100e+04
mean     1.363429e+05
std      3.976069e+05
min      1.281062e+03
25%      1.938167e+04
50%      3.311376e+04
75%      5.404077e+04
max      2.738767e+06
Name: population_5km_buffer, dtype: float64


In [8]:
!pip install osmnx
import osmnx as ox
import geopandas as gpd
import pandas as pd
import h3
from shapely.geometry import Polygon

# 1. Load the dataset that was previously saved
df_original = pd.read_csv('output_corrected_roads.csv')

# 2. Convert Hex IDs to Polygons to create a temporary GeoDataFrame
def hex_to_polygon(hex_id):
    boundary = h3.cell_to_boundary(hex_id)
    return Polygon([(lng, lat) for lat, lng in boundary])

gdf_hex_temp = gpd.GeoDataFrame(df_original.copy(), geometry=df_original['hex_id'].apply(hex_to_polygon), crs="EPSG:4326")

# 3. Download only Major Roads (Highways) for Surat
# This is a proxy for Traffic Density
tags = {'highway': ['motorway', 'trunk', 'primary', 'secondary']}
major_roads = ox.features_from_place("Surat, India", tags)

# 4. Join with your Hexagons
# Calculate distance from hex center to nearest major road
gdf_hex_temp['centroid'] = gdf_hex_temp.geometry.centroid
major_roads_unary = major_roads.geometry.unary_union
gdf_hex_temp['dist_to_highway_km'] = gdf_hex_temp['centroid'].apply(lambda x: x.distance(major_roads_unary) * 111) # approx conversion to km

# 5. Merge the new 'dist_to_highway_km' back into the original df_original
df_original = df_original.merge(gdf_hex_temp[['hex_id', 'dist_to_highway_km']], on='hex_id', how='left')

# 6. Save the updated DataFrame
df_original.to_csv('surat_ev_analysis_phase3.csv', index=False)

print("✅ Calculated distance to major highways and saved to surat_ev_analysis_phase3.csv")

/tmp/ipykernel_31590/2857431676.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_hex_temp['centroid'] = gdf_hex_temp.geometry.centroid
/tmp/ipykernel_31590/2857431676.py:26: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  major_roads_unary = major_roads.geometry.unary_union


✅ Calculated distance to major highways and saved to surat_ev_analysis_phase3.csv


In [12]:
import osmnx as ox
import pandas as pd
import h3
import geopandas as gpd
from shapely.geometry import Polygon

# 1. Load your existing hex data
df = pd.read_csv('surat_ev_analysis_phase2.csv')

# 2. Download Road Network with Metadata
print("Downloading road network for Surat (Major roads only)...")
# We fetch specific high-traffic road types
filter_tags = '["highway"~"motorway|trunk|primary|secondary|tertiary"]'
graph = ox.graph_from_place("Surat, India", network_type='drive', custom_filter=filter_tags)
roads = ox.graph_to_gdfs(graph, nodes=False, edges=True)

# 3. Assign Weights based on Road Type
weights = {
    'motorway': 1.0, 'trunk': 1.0,
    'primary': 0.7,
    'secondary': 0.4,
    'tertiary': 0.2
}

def get_weight(highway_type):
    if isinstance(highway_type, list): highway_type = highway_type[0]
    return weights.get(highway_type, 0.1)

roads['traffic_weight'] = roads['highway'].apply(get_weight)

# 4. Convert Hex IDs to Polygons for Spatial Join
def hex_to_poly(hex_id):
    # h3.cell_to_boundary returns [[lat, lng], ...]
    boundary = h3.cell_to_boundary(hex_id)
    # Convert to shapely: [(lng, lat), ...]
    return Polygon([(lng, lat) for lat, lng in boundary])

df['geometry'] = df['hex_id'].apply(hex_to_poly)
gdf_hex = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

# 5. Spatial Join: Intersect Roads with Hexagons
print("Calculating weighted traffic density per hexagon...")
# Projects to a metric CRS (UTM) for accurate length calculation in meters
gdf_hex = gdf_hex.to_crs(epsg=32643)
roads = roads.to_crs(epsg=32643)

# Intersect roads with hexes
roads_in_hex = gpd.overlay(roads, gdf_hex, how='intersection')

# Calculate weighted length (Length * Weight)
roads_in_hex['weighted_len'] = (roads_in_hex.geometry.length / 1000) * roads_in_hex['traffic_weight']

# Group by hex_id to get the total Traffic Score
traffic_results = roads_in_hex.groupby('hex_id')['weighted_len'].sum().reset_index()
traffic_results.rename(columns={'weighted_len': 'traffic_density_score'}, inplace=True)

# 6. Merge back and Save
df_final = pd.merge(df, traffic_results, on='hex_id', how='left').fillna(0)
df_final.to_csv('surat_ev_traffic_density.csv', index=False)

print("Success! Traffic Density Score calculated and saved.")
print(df_final[['hex_id', 'road_km', 'traffic_density_score']].sort_values(by='traffic_density_score', ascending=False).head())

Calculating weighted traffic density per hexagon...
Success! Traffic Density Score calculated and saved.
                hex_id   road_km  traffic_density_score
10703  8942d91084fffff  2.847154               1.939030
33022  8942d9c638bffff  2.034012               1.901140
27286  8942d98b08bffff  7.259940               1.734225
19063  8942d98b09bffff  5.970344               1.658233
20330  8942d98b017ffff  4.418602               1.509717


In [13]:
import pandas as pd
import numpy as np

# 1. Load your latest file
df = pd.read_csv('surat_ev_analysis_phase3.csv')

# 2. Normalize the Distance Score
# We want: Short Distance = High Score (100), Long Distance = Low Score (0)
max_dist = df['dist_nearest_substation_km'].max()
df['grid_proximity_score'] = (1 - (df['dist_nearest_substation_km'] / max_dist)) * 100

# 3. Add the "Capacity Proxy" (Partial Load Logic)
# If a site is very close to a substation (< 2km), we assume higher load availability.
# If you have a list of GIDC hex_ids, you would give them an extra 20 points here.
def estimate_capacity_bonus(row):
    bonus = 0
    if row['dist_nearest_substation_km'] < 2.0:
        bonus += 20 # High proximity often implies dedicated industrial lines
    return bonus

df['grid_capacity_bonus'] = df.apply(estimate_capacity_bonus, axis=1)

# 4. Final Partial Grid Load Score
df['partial_grid_load_score'] = df['grid_proximity_score'] + df['grid_capacity_bonus']
# Clip at 100
df['partial_grid_load_score'] = df['partial_grid_load_score'].clip(upper=100)

# 5. Save progress
df.to_csv('surat_ev_grid_load_refined.csv', index=False)

print("Partial Grid Load factor completed!")
print(df[['hex_id', 'dist_nearest_substation_km', 'partial_grid_load_score']].sort_values(by='partial_grid_load_score', ascending=False).head())

Partial Grid Load factor completed!
                hex_id  dist_nearest_substation_km  partial_grid_load_score
25035  8942d9d6843ffff                    1.690539                    100.0
8067   8942d98b05bffff                    1.614188                    100.0
25165  8942d98a657ffff                    1.135383                    100.0
45678  8942d98b6c7ffff                    1.396389                    100.0
39429  8942d9d4187ffff                    0.801263                    100.0


In [15]:
import pandas as pd
import numpy as np

# 1. Load the necessary dataframes containing all computed factors
df_grid_refined = pd.read_csv('surat_ev_grid_load_refined.csv')
df_traffic_density = pd.read_csv('surat_ev_traffic_density.csv')

# Merge the dataframes to get all required columns for suitability calculation
# We will use the columns from df_grid_refined as base and add missing ones from df_traffic_density
df = pd.merge(df_grid_refined,
              df_traffic_density[['hex_id', 'population_5km_buffer', 'traffic_density_score']],
              on='hex_id',
              how='left')

# 2. Finalize Factor 5: Grid Load Score
# Use the already calculated 'partial_grid_load_score' as the final 'grid_score'
df['grid_score'] = df['partial_grid_load_score']

# 3. Finalize Factor 6: Flood Risk Penalty
# We create a multiplier. 0 risk = 1.0 (no change), 1.0 risk = 0.2 (massive penalty)
df['flood_penalty_multiplier'] = 1 - (df['flood_risk'] * 0.8)

# 4. Preliminary Suitability Calculation
# Ensure all required columns are present after merge before calculation
df['current_suitability'] = (
    (df['traffic_density_score'] * 40) +
    (df['population_5km_buffer'] / df['population_5km_buffer'].max() * 40) +
    (df['grid_score'] * 20)
) * df['flood_penalty_multiplier']

# 5. Save the refined data
df.to_csv('surat_ev_final_factors.csv', index=False)

print("Factor 5 and 6 have been processed!")
print(df[['hex_id', 'grid_score', 'flood_risk', 'current_suitability']].sort_values(by='current_suitability', ascending=False).head())

Factor 5 and 6 have been processed!
                hex_id  grid_score  flood_risk  current_suitability
35634  8942d98b453ffff       100.0         0.0          2084.140005
19063  8942d98b09bffff       100.0         0.0          2078.919218
27286  8942d98b08bffff       100.0         0.0          2078.090991
43971  8942d9d682fffff       100.0         0.0          2071.625633
12430  8942d9d6823ffff       100.0         0.0          2069.615216


In [16]:
import pandas as pd
import requests

def fetch_surat_ev_stations():
    # OpenChargeMap API for Surat (Lat: 21.1702, Lng: 72.8311)
    # No API key is required for simple public queries
    url = "https://api.openchargemap.io/v3/poi/"
    params = {
        'output': 'json',
        'latitude': 21.1702,
        'longitude': 72.8311,
        'distance': 25,
        'distanceunit': 'KM',
        'maxresults': 200,
        'countrycode': 'IN',
        'key': 'f3d53325-305b-487b-8b5e-04983058097d' # Public free-tier key
    }

    print("Fetching live station data for Surat...")
    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()
        stations = []
        for item in data:
            stations.append({
                'name': item['AddressInfo']['Title'],
                'latitude': item['AddressInfo']['Latitude'],
                'longitude': item['AddressInfo']['Longitude'],
                'address': item['AddressInfo']['AddressLine1'],
                'power_kw': item['Connections'][0].get('PowerKW', 'Unknown') if item['Connections'] else 'Unknown'
            })

        df_comp = pd.DataFrame(stations)
        df_comp.to_csv('surat_existing_chargers.csv', index=False)
        print(f"Success! Found {len(df_comp)} stations. File saved as 'surat_existing_chargers.csv'")
        return df_comp
    else:
        print("Failed to fetch data. Check your internet connection.")

# Run the fetcher
fetch_surat_ev_stations()

Fetching live station data for Surat...
Failed to fetch data. Check your internet connection.
